# CalCOFI DIC 回帰のハイパーパラメータ探索

Optuna の TPE で、5,000エポック目の検証MSEを最小化します。32試行をシード42で学習し、探索上位3設定と現行設定をシード42・123・2026の平均で比較します。結果は `output/optuna/` に保存します。

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from joblib import Parallel, delayed, parallel_config
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("output/optuna")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 42
SEARCH_SEED = 42
CONFIRMATION_SEEDS = (SEARCH_SEED, 123, 2026)
MAX_EPOCHS = 5000
N_TRIALS = 32
N_STARTUP_TRIALS = 8
N_WORKERS = 4
N_FINALISTS = 3
VAL_BATCH_SIZE = 64
MIN_LR = 1e-6

CATEGORICAL_CHOICES = {
    "hidden1": (64, 128, 256),
    "hidden2": (32, 64, 128),
    "celu_alpha": (1, 2, 3, 5),
    "huber_delta": (5, 10, 20),
    "batch_size": (32, 64, 128),
}
NUMERIC_RANGES = {
    "learning_rate": (1e-4, 3e-3),
    "weight_decay": (1e-5, 1e-1),
}
BASELINE = {
    "hidden1": 128, "hidden2": 64, "celu_alpha": 3,
    "learning_rate": 0.001, "weight_decay": 0.025,
    "huber_delta": 10, "batch_size": 64,
}

### データ分割と前処理

提出用ノートブックに合わせ、平均補完は訓練分割で適合させ、標準化は補完後の訓練CSV全行（検証分割の特徴量を含む）で適合させます。検証分割のDICは評価に使います。

In [ ]:
train_csv = DATA_DIR / "train.csv"
train = pd.read_csv(train_csv).drop(columns=["Unnamed: 12", "id"])
test_columns = pd.read_csv(DATA_DIR / "test.csv", nrows=0).columns
feature_columns = train.drop(columns="DIC").columns.intersection(test_columns)
X_train, X_val, y_train, y_val = train_test_split(
    train[feature_columns], train["DIC"], test_size=0.2, random_state=SPLIT_SEED,
)

imputer = SimpleImputer(strategy="mean").fit(X_train)
X_train = imputer.transform(X_train)
X_val = imputer.transform(X_val)
scaler = StandardScaler().fit(np.vstack([X_train, X_val]))
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
y_train = y_train.to_numpy(dtype=np.float32)
y_val = y_val.to_numpy(dtype=np.float32)

print(f"Training: {len(y_train)} rows, validation: {len(y_val)} rows, features: {len(feature_columns)}")
print(f"StandardScaler fit: {int(scaler.n_samples_seen_)} rows")

In [ ]:
def fit_and_score(params, seed, X_train, X_val, y_train, y_val):
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    torch.manual_seed(seed)

    train_data = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32),
    )
    val_data = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32),
    )
    train_loader = DataLoader(
        train_data, batch_size=params["batch_size"], shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    val_loader = DataLoader(val_data, batch_size=VAL_BATCH_SIZE)
    model = nn.Sequential(
        nn.Linear(X_train.shape[1], params["hidden1"]),
        nn.CELU(alpha=params["celu_alpha"]),
        nn.Linear(params["hidden1"], params["hidden2"]),
        nn.CELU(alpha=params["celu_alpha"]),
        nn.Linear(params["hidden2"], 1),
    )
    criterion = nn.HuberLoss(delta=params["huber_delta"])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LR,
    )

    model.train()
    for _ in range(MAX_EPOCHS):
        for features, targets in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(features).squeeze(-1), targets)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        predictions = torch.cat([model(features).squeeze(-1) for features, _ in val_loader])
        return nn.functional.mse_loss(predictions, val_data.tensors[1]).item()


def suggest_params(trial):
    params = {
        name: trial.suggest_categorical(name, choices)
        for name, choices in CATEGORICAL_CHOICES.items()
    }
    params.update({
        name: trial.suggest_float(name, lower, upper, log=True)
        for name, (lower, upper) in NUMERIC_RANGES.items()
    })
    return params

### TPE 探索と上位設定の再評価

現行設定を試行0に登録します。最初の8試行（現行設定1件とランダム7件）の後、TPEで24試行を提案します。4件ずつ提案・並列学習し、試行番号順に結果を登録します。学習率と weight decay は対数スケールで探索します。

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=SEARCH_SEED, n_startup_trials=N_STARTUP_TRIALS, multivariate=True,
)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.enqueue_trial(BASELINE)
training_data = (X_train, X_val, y_train, y_val)

with parallel_config(backend="loky", n_jobs=N_WORKERS, inner_max_num_threads=1):
    with Parallel() as run_models:
        for start in range(0, N_TRIALS, N_WORKERS):
            trials = [study.ask() for _ in range(min(N_WORKERS, N_TRIALS - start))]
            candidates = [suggest_params(trial) for trial in trials]
            scores = run_models(
                delayed(fit_and_score)(params, SEARCH_SEED, *training_data)
                for params in candidates
            )
            for trial, score in zip(trials, scores):
                study.tell(trial, score)
                print(f"Trial {trial.number:02d}, final validation MSE = {score:.6f}", flush=True)
            study.trials_dataframe(attrs=("number", "value", "params", "state")).to_csv(
                OUTPUT_DIR / "trials.csv", index=False,
            )

        baseline_trial = study.trials[0]
        ranked_trials = sorted(
            (trial for trial in study.trials if trial.number != 0 and trial.value is not None),
            key=lambda trial: trial.value,
        )
        finalists = [baseline_trial, *ranked_trials[:N_FINALISTS]]
        extra_scores = iter(run_models(
            delayed(fit_and_score)(trial.params, seed, *training_data)
            for trial in finalists
            for seed in CONFIRMATION_SEEDS[1:]
        ))

results = []
for trial in finalists:
    scores = [trial.value, *(next(extra_scores) for _ in CONFIRMATION_SEEDS[1:])]
    results.append({
        "trial": trial.number,
        **trial.params,
        **{f"val_mse_seed_{seed}": score for seed, score in zip(CONFIRMATION_SEEDS, scores)},
        "mean_final_val_mse": float(np.mean(scores)),
        "std_final_val_mse": float(np.std(scores, ddof=1)),
    })
results.sort(key=lambda row: row["mean_final_val_mse"])
finalist_results = pd.DataFrame(results)
finalist_results.to_csv(OUTPUT_DIR / "finalists.csv", index=False)

adopted = results[0]
adopted_trial = study.trials[adopted["trial"]]
summary = {
    "optuna_version": optuna.__version__,
    "torch_version": torch.__version__,
    "sampler": "multivariate TPE",
    "trials": N_TRIALS,
    "startup_trials": N_STARTUP_TRIALS,
    "workers": N_WORKERS,
    "search_seed": SEARCH_SEED,
    "confirmation_seeds": CONFIRMATION_SEEDS,
    "split_seed": SPLIT_SEED,
    "epochs": MAX_EPOCHS,
    "validation_batch_size": VAL_BATCH_SIZE,
    "cosine_eta_min": MIN_LR,
    "scaler_fit_rows": int(scaler.n_samples_seen_),
    "data_sha256": hashlib.sha256(train_csv.read_bytes()).hexdigest(),
    "best_search_trial": study.best_trial.number,
    "adopted_trial": adopted_trial.number,
    "adopted_params": adopted_trial.params,
    "adopted_mean_final_val_mse": adopted["mean_final_val_mse"],
    "adopted_std_final_val_mse": adopted["std_final_val_mse"],
    "adopted_seed_42_final_val_mse": adopted_trial.value,
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8",
)
finalist_results

### パラメータ別の検証MSEと重要度

点は探索時（シード42）の実測値です。赤いひし形は3シードの再評価で採用した設定です。重要度は同じ探索結果から fANOVA で推定します。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from optuna.importance import FanovaImportanceEvaluator, get_param_importances
from optuna.trial import TrialState

completed_trials = study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
parameter_labels = {
    "hidden1": "Hidden layer 1",
    "hidden2": "Hidden layer 2",
    "celu_alpha": "CELU alpha",
    "learning_rate": "Initial learning rate",
    "weight_decay": "Weight decay",
    "huber_delta": "Huber delta",
    "batch_size": "Training batch size",
}

fig, axes = plt.subplots(4, 2, figsize=(11, 11), sharey=True, layout="constrained")
for ax, (name, label) in zip(axes.flat, parameter_labels.items()):
    parameter_values = [trial.params[name] for trial in completed_trials]
    adopted_value = adopted_trial.params[name]
    if name in CATEGORICAL_CHOICES:
        choices = list(CATEGORICAL_CHOICES[name])
        positions = {choice: index for index, choice in enumerate(choices)}
        parameter_values = [positions[value] for value in parameter_values]
        adopted_value = positions[adopted_value]
        ax.set_xticks(range(len(choices)), labels=[str(choice) for choice in choices])
        ax.set_xlim(-0.35, len(choices) - 0.65)
    else:
        ax.set_xscale("log")
        ax.set_xlim(*NUMERIC_RANGES[name])
        ax.xaxis.set_minor_formatter(NullFormatter())
        if name == "learning_rate":
            ax.set_xticks([1e-4, 3e-4, 1e-3, 3e-3], labels=["1e-4", "3e-4", "1e-3", "3e-3"])

    trials_artist = ax.scatter(
        parameter_values, [trial.value for trial in completed_trials],
        color="#2563a0", alpha=0.52, s=24, label="Completed study trial",
    )
    adopted_artist = ax.scatter(
        adopted_value, adopted_trial.value, color="#bd4a26", edgecolors="white",
        linewidths=0.7, marker="D", s=66, zorder=3, label="Adopted setting in study",
    )
    ax.set_title(label, loc="left")
    ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.25)

axes[-1, -1].axis("off")
axes[-1, -1].legend(handles=[trials_artist, adopted_artist], loc="center", frameon=False)
fig.suptitle("Optuna trial results by hyperparameter", fontsize=16)
fig.supylabel("Validation MSE (log scale)")
fig.savefig(OUTPUT_DIR / "parameter_slices.png", dpi=160)
plt.show()
plt.close(fig)

importances = get_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=42))
names = list(importances)
scores = [importances[name] for name in names]

fig, ax = plt.subplots(figsize=(8, 4.6), layout="constrained")
ax.barh([parameter_labels[name] for name in names], scores, color="#2563a0")
ax.invert_yaxis()
ax.set_xlim(0, min(1.1, 1.18 * max(scores)))
ax.bar_label(ax.containers[0], fmt="%.3f", padding=4)
ax.set_title("Optuna hyperparameter importance (fANOVA)", loc="left")
ax.set_xlabel("Normalized importance")
ax.set_axisbelow(True)
ax.grid(axis="x", alpha=0.25)
fig.savefig(OUTPUT_DIR / "parameter_importance.png", dpi=160)
plt.show()
plt.close(fig)
